# Sequence models: nn.RNN, nn.LSTM, nn.GRU

_PyTorch_

**PyTorch modules that read an ordered sequence one step at a time and carry a hidden state forward.**

A recurrent module is a small function applied in a loop. It keeps a hidden state — a vector that summarizes everything seen so far — and at each timestep it combines the new input with the old hidden state to produce a new hidden state. PyTorch runs that loop for you over the whole sequence in one call; you just hand it the data and read off the results.

---

This notebook is a practice scaffold for the **Sequence models: nn.RNN, nn.LSTM, nn.GRU** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # reproducible

# ---- SYNTHETIC TASK: is each sequence trending UP (1) or DOWN (0)? ----
# Each sample: a length-SEQ_LEN sequence of 1-feature values.
# Up sequences rise on average; down sequences fall. Plus noise so it is not trivial.
N, SEQ_LEN, FEAT = 600, 12, 1
HIDDEN, N_CLASSES = 16, 2

def make_batch(n):
    y = torch.randint(0, 2, (n,))                 # class label per sequence: 0 or 1
    slope = torch.where(y == 1, 0.3, -0.3)        # up -> +slope, down -> -slope
    t = torch.arange(SEQ_LEN).float()             # 0,1,...,SEQ_LEN-1
    base = slope[:, None] * t[None, :]            # (n, SEQ_LEN): the trend line
    noise = 0.5 * torch.randn(n, SEQ_LEN)         # make it noisy
    x = (base + noise).unsqueeze(-1)              # (n, SEQ_LEN, FEAT)  <- batch_first layout
    return x, y

X_train, y_train = make_batch(N)
X_test,  y_test  = make_batch(200)
print("input shape (batch, seq_len, features):", tuple(X_train.shape))  # (600, 12, 1)

# ---- MODEL: LSTM -> take LAST hidden state -> Linear classifier ----
class SeqClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # batch_first=True => input is (batch, seq_len, features). The classic shape fix.
        self.lstm = nn.LSTM(input_size=FEAT, hidden_size=HIDDEN, batch_first=True)
        self.fc   = nn.Linear(HIDDEN, N_CLASSES)  # logits; CrossEntropyLoss adds softmax

    def forward(self, x):
        # output: (batch, seq_len, HIDDEN) -- hidden state at EVERY step
        # h_n:    (num_layers, batch, HIDDEN) -- hidden state at the LAST step only
        output, (h_n, c_n) = self.lstm(x)
        last = h_n[-1]            # (batch, HIDDEN): top layer's final hidden state
        return self.fc(last)      # (batch, N_CLASSES): raw logits

model = SeqClassifier()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()  # expects raw logits + integer class indices

# ---- TRAIN a few epochs ----
model.train()
for epoch in range(15):
    opt.zero_grad()                 # clear old grads (they accumulate otherwise)
    logits = model(X_train)         # forward
    loss = loss_fn(logits, y_train)
    loss.backward()                 # backprop through time
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # tame exploding grads
    opt.step()
    if epoch % 3 == 0:
        print(f"epoch {epoch:2d}  loss {loss.item():.3f}")

# ---- EVALUATE: accuracy on held-out sequences ----
model.eval()
with torch.no_grad():               # no graph at inference -> saves memory
    pred = model(X_test).argmax(dim=1)
    acc = (pred == y_test).float().mean().item()
print(f"test accuracy: {acc:.3f}")  # ~0.9+ on this easy synthetic task

# ---- shapes, made explicit ----
with torch.no_grad():
    out, (h_n, c_n) = model.lstm(X_test[:4])   # peek at a 4-sequence batch
print("output (every step):", tuple(out.shape))    # (4, 12, 16)
print("h_n (last step)    :", tuple(h_n.shape))    # (1, 4, 16)
print("h_n[-1] -> Linear  :", tuple(h_n[-1].shape))# (4, 16)


## Visualize the data & results

_Why do plain RNNs forget? How a gated cell keeps a signal alive over long sequences._

In [ ]:
import numpy as np

# A plain RNN's recurrent weight shrinks the hidden state each step; a gated cell's
# forget gate sits near 1, so its 'conveyor belt' carries the signal far longer.
T = np.arange(0, 41, 2)          # timesteps 0, 2, ..., 40
rnn  = 0.7 ** T                  # plain RNN decay factor 0.7
lstm = 0.98 ** T                 # LSTM/GRU near-constant carry, factor 0.98

for t, r, l in zip(T, rnn, lstm):
    print(f"t={t:2d}  rnn={r:.4f}  lstm={l:.4f}")
# t= 0  rnn=1.0000  lstm=1.0000
# t=20  rnn=0.0008  lstm=0.6676   <- RNN has forgotten; LSTM still remembers
# t=40  rnn=0.0000  lstm=0.4457

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Build rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True). Feed it x = torch.randn(2, 4, 3) (2 sequences, 4 steps, 3 features). Call output, h_n = rnn(x). Predict the shapes of output and h_n before running, then print both.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Construct with batch_first=True so input is (batch, seq_len, features). — _It matches the (2, 4, 3) tensor you are passing._
- Unpack output, h_n = rnn(x) and print each .shape. — _output is every step (batch, seq, hidden); h_n is the last step (num_layers, batch, hidden)._

**Answer:** import torch
import torch.nn as nn
torch.manual_seed(0)
rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)
x = torch.randn(2, 4, 3)
output, h_n = rnn(x)
print(output.shape)   # torch.Size([2, 4, 5])  hidden at every step
print(h_n.shape)      # torch.Size([1, 2, 5])  last step only

</details>

**Problem 2.** Type this in Colab. THE classic RNN shape bug. Build lstm = nn.LSTM(10, 20) with the defaults (no batch_first). You have data shaped (32, 15, 10) = 32 sequences, length 15, 10 features. First feed it raw and print h_n.shape — notice the batch dimension is wrong. Then rebuild with batch_first=True and confirm h_n reports 32 sequences.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With the default layout, the module reads axis 0 as seq_len and axis 1 as batch. — _So (32, 15, 10) is misread as 32 steps of a 15-sequence batch — silently wrong._
- Rebuild with nn.LSTM(10, 20, batch_first=True). — _Now axis 0 is the batch, so h_n correctly reports 32 sequences._

**Answer:** x = torch.randn(32, 15, 10)
bad = nn.LSTM(10, 20)                      # default: (seq, batch, feat)
_, (h_n, _) = bad(x)
print(h_n.shape)   # torch.Size([1, 15, 20])

</details>

**Problem 3.** Type this in Colab. Build lstm = nn.LSTM(input_size=3, hidden_size=8, batch_first=True) and feed x = torch.randn(4, 6, 3). Unpack the LSTM's return correctly (it bundles hidden and cell state in a tuple), then take the top layer's last hidden state and print its shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Unpack as output, (h_n, c_n) = lstm(x). — _An LSTM returns (h_n, c_n) as a tuple, unlike RNN/GRU which return just h_n._
- Take h_n[-1] for the top layer's final hidden state. — _That single vector per sequence is what a classifier head consumes._

**Answer:** torch.manual_seed(0)
lstm = nn.LSTM(input_size=3, hidden_size=8, batch_first=True)
x = torch.randn(4, 6, 3)
output, (h_n, c_n) = lstm(x)
print(output.shape)    # torch.Size([4, 6, 8])  every step
print(h_n.shape)       # torch.Size([1, 4, 8])  last step
last = h_n[-1]
print(last.shape)      # torch.Size([4, 8])  -> feed to nn.Linear

</details>

**Problem 4.** Type this in Colab. One label per whole sequence. Reuse the LSTM above (hidden_size=8), add fc = nn.Linear(8, 3) for 3 classes, run x = torch.randn(4, 6, 3) through both, and print the logits shape. Use h_n[-1] — the last timestep — not output.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass h_n[-1] (shape (4, 8)) into fc. — _A single label per sequence needs one summary vector, not one per step._
- Print the resulting logits shape. — _It should be (batch, num_classes) = (4, 3), ready for CrossEntropyLoss._

**Answer:** torch.manual_seed(0)
lstm = nn.LSTM(3, 8, batch_first=True)
fc = nn.Linear(8, 3)
x = torch.randn(4, 6, 3)
_, (h_n, _) = lstm(x)
logits = fc(h_n[-1])        # last timestep -> classifier
print(logits.shape)   # torch.Size([4, 3])  raw logits, one row per sequence

</details>

**Problem 5.** Type this in Colab. Tokens in. Build emb = nn.Embedding(num_embeddings=10, embedding_dim=4) and a batch of token ids ids = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]]) (shape (2, 4)). Embed it and print the new shape, then feed it to nn.GRU(4, 6, batch_first=True) and print the GRU output shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply emb(ids) to map each id to a length-4 vector. — _It turns (batch, seq) of ids into (batch, seq, embed_dim) for the recurrent layer._
- Pass the embeddings to the GRU and unpack output, h_n. — _A GRU returns just h_n (no cell state), unlike the LSTM._

**Answer:** torch.manual_seed(0)
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)
ids = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
e = emb(ids)
print(e.shape)        # torch.Size([2, 4, 4])  (batch, seq, embed_dim)
gru = nn.GRU(4, 6, batch_first=True)
output, h_n = gru(e)
print(output.shape)   # torch.Size([2, 4, 6])
print(h_n.shape)      # torch.Size([1, 2, 6])

</details>

**Problem 6.** Type this in Colab. Embedding index out of range. Build emb = nn.Embedding(num_embeddings=5, embedding_dim=3) — valid ids are 0..4. Run emb(torch.tensor([5])) and observe the error. Then fix it by sizing the table to num_embeddings=6 and re-run on the same id.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look up an id equal to num_embeddings. — _Row indices only go 0 .. num_embeddings-1, so id 5 is out of range and raises an IndexError._
- Enlarge the table to cover your full vocabulary. — _num_embeddings must exceed the largest token id (leave slots for padding/unknown too)._

**Answer:** # Broken: IndexError -- index 5 is out of bounds for a table of size 5
# emb = nn.Embedding(num_embeddings=5, embedding_dim=3)
# emb(torch.tensor([5]))

torch.manual_seed(0)
emb = nn.Embedding(num_embeddings=6, embedding_dim=3)   # ids 0..5 now valid
print(emb(torch.tensor([5])).shape)   # torch.Size([1, 3])

</details>

**Problem 7.** Type this in Colab. Variable lengths. You have 3 sequences of lengths 5, 9, 3 with 2 features each, all padded to length 9 — build x = torch.zeros(3, 9, 2) and lengths = torch.tensor([5, 9, 3]). Pack it, run an LSTM, unpack, and confirm the output comes back at shape (3, 9, hidden).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap with pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False). — _Packing tells the LSTM each true length so it skips padding and h_n reflects the real last token._
- Run the LSTM on the packed input, then pad_packed_sequence(out, batch_first=True). — _It restores the padded rectangular shape for downstream layers._

**Answer:** from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
torch.manual_seed(0)
x = torch.zeros(3, 9, 2)
lengths = torch.tensor([5, 9, 3])
lstm = nn.LSTM(2, 4, batch_first=True)
packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
out_packed, (h_n, c_n) = lstm(packed)
out, lens = pad_packed_sequence(out_packed, batch_first=True)
print(out.shape)    # torch.Size([3, 9, 4])
print(lens)         # tensor([5, 9, 3])  true lengths preserved

</details>

**Problem 8.** Type this in Colab. Tiny end-to-end sequence classifier. Generate x = torch.randn(8, 5, 2) and y = torch.randint(0, 2, (8,)). Build an LSTM(2 &rarr; 16, batch_first) plus nn.Linear(16, 2), run one training step with CrossEntropyLoss and Adam (remember zero_grad and gradient clipping), and print the loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward: take h_n[-1] into the linear head to get logits, score with CrossEntropyLoss. — _Cross-entropy wants raw logits and integer class indices — no softmax._
- zero_grad(), backward(), clip_grad_norm_(..., 1.0), step(). — _Clipping tames exploding gradients through time; zeroing prevents accumulation._

**Answer:** torch.manual_seed(0)
x = torch.randn(8, 5, 2)
y = torch.randint(0, 2, (8,))
lstm = nn.LSTM(2, 16, batch_first=True)
fc = nn.Linear(16, 2)
params = list(lstm.parameters()) + list(fc.parameters())
opt = torch.optim.Adam(params, lr=0.01)
loss_fn = nn.CrossEntropyLoss()

opt.zero_grad()
_, (h_n, _) = lstm(x)
logits = fc(h_n[-1])          # (8, 2) raw logits
loss = loss_fn(logits, y)
loss.backward()
nn.utils.clip_grad_norm_(params, 1.0)
opt.step()
print(round(loss.item(), 4))   # ~0.70  (near ln(2), random start)

</details>